# Практическое занятие: решающие деревья и композиции моделей

## Темы

- решающие деревья для классификации и регрессии;
- bootstrap и bagging;
- random subspaces (случайные подпространства признаков);
- случайный лес;
- out-of-bag (OOB) оценка;
- подбор гиперпараметров с `GridSearchCV`;
- интерпретация важности признаков.

Ноутбук содержит **5 задач на 5 разных встроенных датасетах scikit-learn**. Интернет для загрузки данных не нужен.

### Результаты обучения

После занятия студент должен уметь:

1. строить дерево классификации и дерево регрессии;
2. объяснять переобучение дерева через его глубину и размер листа;
3. отличать bootstrap по объектам от random subspaces по признакам;
4. объяснять, почему ансамблю полезны разнообразные деревья;
5. корректно применять `Pipeline`, `GridSearchCV`, CV и отдельную test-выборку;
6. вычислять и интерпретировать OOB-оценку;
7. сравнивать impurity importance и permutation importance.

### Важные правила эксперимента

- Везде используем `random_state=42` для воспроизводимости.
- Test-выборка не участвует в выборе гиперпараметров.
- Для классификации применяем стратифицированное разбиение.
- Масштабирование деревьям обычно **не требуется**: дерево сравнивает значения с порогом, поэтому монотонное изменение масштаба не меняет порядок объектов. В `Pipeline` оставляем обучаемую обработку пропусков, чтобы показать корректную архитектуру без утечки.
- Важность признака означает связь с прогнозом модели, но не доказывает причинность.

### Ключевые идеи

| Метод | Случайность по объектам | Случайность по признакам | Что усредняем |
|---|---:|---:|---|
| Одно дерево | нет | нет | одну модель |
| Bagging | обычно bootstrap | необязательно | много деревьев |
| Random subspaces | необязательно | подмножество признаков для каждой модели | много моделей |
| Случайный лес | обычно bootstrap | случайное подмножество признаков в каждом разбиении | много деревьев |

Для bootstrap-выборки размера \(n\), взятой с возвращением из \(n\) объектов, доля уникальных объектов стремится к

\[
1 - \left(1-\frac{1}{n}\right)^n \approx 1-e^{-1}\approx 0.632.
\]

Оставшиеся примерно 36.8% объектов являются OOB для конкретного дерева.

## Система оценивания: 100 баллов

Каждая задача оценивается максимум в 20 баллов:

| Компонент | Баллы |
|---|---:|
| Загрузка данных и корректная постановка | 2 |
| EDA и содержательная визуализация | 3 |
| Разбиение без утечки, обоснование preprocessing, `Pipeline` | 3 |
| Осмысленная сетка и корректный `GridSearchCV` только на train | 4 |
| Прогноз и подходящие метрики | 3 |
| Важность признаков и интерпретация | 3 |
| Ответы на контрольные вопросы | 2 |

Ориентиры: 90–100 — отлично; 75–89 — хорошо; 55–74 — удовлетворительно; менее 55 — требуется доработка. Оценивается логика исследования, а не достижение заранее заданной метрики.

## Подготовка окружения

Нужны Python 3.10+ и библиотеки `numpy`, `pandas`, `matplotlib`, `seaborn`, `scikit-learn`, `jupyter`.

Если библиотек нет, раскомментируйте следующую строку и выполните её один раз:

```python
# %pip install -U numpy pandas matplotlib seaborn "scikit-learn>=1.4,<2.0" jupyter
```

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from IPython.display import display

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.datasets import (
    load_breast_cancer,
    load_diabetes,
    load_digits,
    load_iris,
    load_wine,
)
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    RocCurveDisplay,
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    StratifiedKFold,
    cross_val_predict,
    cross_val_score,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.titlesize"] = 13

RANDOM_STATE = 42

print("scikit-learn:", sklearn.__version__)
print("numpy:", np.__version__)
print("pandas:", pd.__version__)

## Шаблон работы над каждой задачей

В кодовых ячейках ниже заменяйте `TODO` своим кодом. После каждой задачи добавьте краткий вывод (4–7 предложений):

- лучшая конфигурация и CV-метрика;
- итоговые test-метрики;
- признаки/пиксели, оказавшиеся наиболее важными;
- признаки переобучения или недообучения;
- ответы на контрольные вопросы.

Не удаляйте выводы ячеек перед сдачей: они нужны для проверки воспроизводимости.

# Задача 1. Классификация ирисов одним решающим деревом

**Датасет:** `load_iris` — 150 цветков, 4 числовых признака, 3 класса.

**Цель:** построить интерпретируемое дерево классификации и исследовать влияние его сложности на переобучение.

## Что требуется

1. Загрузить данные в `DataFrame`; проверить размер, типы, пропуски, дубликаты и баланс классов.
2. Построить не менее двух визуализаций, одна из них должна показывать связь признаков с классом.
3. Разделить данные на train/test в пропорции 80/20 со `stratify=y`.
4. Объяснить, почему масштабирование не нужно.
5. Собрать `Pipeline`: `SimpleImputer` → `DecisionTreeClassifier`.
6. Подобрать `criterion`, `max_depth`, `min_samples_leaf`, `ccp_alpha` через `GridSearchCV` и стратифицированную 5-fold CV. Основная метрика — `f1_macro`.
7. Получить прогнозы; вывести accuracy, macro-F1, `classification_report` и confusion matrix.
8. Визуализировать верхние уровни дерева и ранжировать признаки по `feature_importances_`.
9. Сформулировать краткий вывод о переобучении и важных признаках.

## Контрольные вопросы

1. Почему стандартизация почти не влияет на дерево?
2. Как `max_depth` и `min_samples_leaf` управляют переобучением?
3. Зачем при трёх классах смотреть macro-F1 и confusion matrix, а не только accuracy?
4. Как прочитать один путь от корня дерева до листа?

In [ ]:
# 1.1 Загрузка и первичная проверка
iris = load_iris(as_frame=True)
X1, y1 = iris.data.copy(), iris.target.copy()

display(X1.head())
print("Размер X:", X1.shape)
print("Названия классов:", iris.target_names)

# TODO: типы, пропуски, дубликаты, баланс классов.

In [ ]:
# 1.2 EDA и визуализация
# TODO: создайте таблицу для визуализации с текстовыми названиями классов.
# TODO: постройте минимум два графика.

In [ ]:
# 1.3 Split, Pipeline и GridSearchCV
# TODO: train_test_split(..., stratify=y1, random_state=RANDOM_STATE)
# TODO: Pipeline([("imputer", ...), ("model", DecisionTreeClassifier(...))])
# TODO: сетка criterion, max_depth, min_samples_leaf, ccp_alpha.
# TODO: StratifiedKFold и GridSearchCV(scoring="f1_macro").

In [ ]:
# 1.4 Итоговая оценка и интерпретация
# TODO: predict; accuracy, macro-F1, classification_report, confusion matrix.
# TODO: сравните train и CV/test, покажите верхние уровни дерева.
# TODO: таблица/график feature_importances_.

### Вывод и ответы к задаче 1

**Ваш вывод:** …

1. …
2. …
3. …
4. …

# Задача 2. Регрессия прогрессирования диабета

**Датасет:** `load_diabetes` — 442 наблюдения, 10 числовых признаков, непрерывная целевая переменная.

**Цель:** построить дерево регрессии, сравнить его с наивной baseline-моделью и проанализировать остатки.

## Что требуется

1. Загрузить данные; проверить структуру, пропуски, дубликаты и распределение target.
2. Построить гистограмму target и график связи одного информативного признака с target.
3. Разделить данные на train/test 80/20.
4. Объяснить, почему дополнительное масштабирование для дерева не требуется. Учтите, что признаки этого датасета уже нормированы при подготовке sklearn.
5. Собрать `Pipeline`: `SimpleImputer` → `DecisionTreeRegressor`.
6. Настроить `criterion`, `max_depth`, `min_samples_leaf`, `max_features` через 5-fold `KFold`. Оптимизировать MSE/RMSE.
7. Сравнить результат с `DummyRegressor(strategy="mean")` по MAE, RMSE и \(R^2\).
8. Построить графики «факт–прогноз» и остатков.
9. Показать важности признаков и верхние уровни дерева.

## Контрольные вопросы

1. Почему \(R^2\) на test может быть отрицательным?
2. Когда MAE предпочтительнее RMSE?
3. Почему глубокое дерево имеет низкое смещение, но высокую дисперсию?
4. Почему нельзя подбирать параметры по test-метрике?

In [ ]:
# 2.1 Загрузка и EDA
diabetes = load_diabetes(as_frame=True)
X2, y2 = diabetes.data.copy(), diabetes.target.copy()

display(X2.head())
print("Размер X:", X2.shape)
print(y2.describe())

# TODO: пропуски, дубликаты, описательная статистика.
# TODO: гистограмма target и график связи информативного признака с target.

In [ ]:
# 2.2 Split, Pipeline и GridSearchCV
# TODO: train_test_split(random_state=RANDOM_STATE).
# TODO: Pipeline с SimpleImputer и DecisionTreeRegressor.
# TODO: сетка criterion, max_depth, min_samples_leaf, max_features.
# TODO: KFold и GridSearchCV(scoring="neg_mean_squared_error").

In [ ]:
# 2.3 Метрики, baseline, остатки и важности
# TODO: DummyRegressor(strategy="mean") и лучшая модель.
# TODO: MAE, RMSE, R² для baseline и дерева.
# TODO: графики факт–прогноз и остатков.
# TODO: feature_importances_ и верхние уровни дерева.

### Вывод и ответы к задаче 2

**Ваш вывод:** …

1. …
2. …
3. …
4. …

# Задача 3. Bootstrap и bagging на данных о винах

**Датасет:** `load_wine` — 178 объектов, 13 химических признаков, 3 класса.

**Цель:** увидеть bootstrap «вручную» и проверить, как bagging меняет стабильность дерева.

## Что требуется

1. Выполнить EDA и построить две содержательные визуализации.
2. Разделить данные 80/20 со стратификацией.
3. На train-части вручную создать bootstrap-выборку размера \(n\):
   - посчитать долю уникальных объектов;
   - найти OOB-объекты;
   - показать распределение числа попаданий объектов в bootstrap.
4. Собрать `Pipeline`: `SimpleImputer` → `BaggingClassifier(DecisionTreeClassifier)`.
5. В `GridSearchCV` сравнить bootstrap и выбор без возвращения (`bootstrap=False`), подобрать `n_estimators`, `max_samples` и параметры базового дерева.
6. Сравнить одно дерево и лучший bagging на одном test-разбиении по balanced accuracy и macro-F1.
7. Построить confusion matrix и получить permutation importance.

## Контрольные вопросы

1. Откуда возникает доля уникальных объектов около \(1-e^{-1}\approx0.632\)?
2. Чем bootstrap отличается от выборки без возвращения?
3. Почему bagging в основном снижает variance, а не bias?
4. Что меняется при росте `n_estimators` и `max_samples`?

In [ ]:
# 3.1 Загрузка и EDA
wine = load_wine(as_frame=True)
X3, y3 = wine.data.copy(), wine.target.copy()

display(X3.head())
print("Размер X:", X3.shape)
print("Классы:", wine.target_names)

# TODO: проверки и две визуализации.

In [ ]:
# 3.2 Split и bootstrap вручную
# TODO: стратифицированное разбиение.
# TODO: сгенерируйте n индексов train с возвращением.
# TODO: доля уникальных, OOB-индексы, график частот попаданий.

In [ ]:
# 3.3 Bagging Pipeline и GridSearchCV
# TODO: BaggingClassifier(estimator=DecisionTreeClassifier(...)).
# Для sklearn < 1.2 параметр назывался base_estimator.
# TODO: сравните bootstrap=True/False, подберите n_estimators,
# max_samples и параметры estimator.

In [ ]:
# 3.4 Сравнение моделей и важности
# TODO: одно дерево и лучший bagging на одном split.
# TODO: balanced accuracy, macro-F1, confusion matrix.
# TODO: permutation_importance для Pipeline.

### Вывод и ответы к задаче 3

**Ваш вывод:** …

1. …
2. …
3. …
4. …

# Задача 4. Random subspaces и случайный лес для распознавания цифр

**Датасет:** `load_digits` — изображения 8×8, 64 пиксельных признака, 10 классов.

**Цель:** сравнить разные источники случайности ансамбля и понять роль `max_features`.

## Что требуется

1. Проверить данные и визуализировать примеры изображений и баланс классов.
2. Разделить данные 80/20 со стратификацией.
3. На одном split сравнить:
   - одно дерево;
   - bagging только по объектам;
   - random subspaces — подмножество признаков для каждого дерева;
   - random patches — случайность и по объектам, и по признакам;
   - `RandomForestClassifier`.
4. Для случайного леса собрать `Pipeline` и подобрать `n_estimators`, `max_features`, `max_depth`, `min_samples_leaf` через `GridSearchCV`.
5. Оценить accuracy, macro-F1 и confusion matrix.
6. Преобразовать 64 важности пикселей обратно в матрицу 8×8 и построить heatmap.
7. Объяснить различие между классическим random subspaces (подмножество на модель) и `max_features` случайного леса (подмножество кандидатов в каждом узле).

## Контрольные вопросы

1. Почему более слабые, но менее коррелированные деревья могут дать лучший ансамбль?
2. Чем `max_features="sqrt"` отличается от использования всех признаков?
3. Почему рост числа деревьев обычно не вызывает классического переобучения, но увеличивает стоимость вычислений?
4. Почему impurity importance может вводить в заблуждение?

In [ ]:
# 4.1 Загрузка и EDA
digits = load_digits(as_frame=True)
X4, y4 = digits.data.copy(), digits.target.copy()

print("Размер X:", X4.shape)
print("Диапазон пикселей:", X4.to_numpy().min(), X4.to_numpy().max())

# TODO: пропуски/дубликаты/баланс и сетка примеров изображений 8×8.

In [ ]:
# 4.2 Сравнение источников случайности
# TODO: единый stratified split.
# TODO: обучите дерево, bagging_objects, random_subspaces,
# random_patches и random_forest.
# TODO: таблица accuracy и macro-F1.

In [ ]:
# 4.3 Pipeline и GridSearchCV для случайного леса
# TODO: Pipeline с imputer и RandomForestClassifier.
# TODO: сетка n_estimators, max_features, max_depth, min_samples_leaf.
# Для экономии времени допустима StratifiedKFold(n_splits=3).

In [ ]:
# 4.4 Метрики и карта важности пикселей
# TODO: test-метрики и confusion matrix.
# TODO: reshape feature_importances_ в (8, 8), heatmap и top-пиксели.

### Вывод и ответы к задаче 4

**Ваш вывод:** …

1. …
2. …
3. …
4. …

# Задача 5. Случайный лес и OOB в медицинской классификации

**Датасет:** `load_breast_cancer` — 569 наблюдений, 30 признаков, два класса.

Для удобной интерпретации перекодируем target: **1 — злокачественная опухоль, 0 — доброкачественная**. Это важно: в исходном датасете sklearn кодировка обратная.

**Цель:** настроить случайный лес, сопоставить CV, OOB и test-оценки и корректно интерпретировать ошибки медицинской модели.

## Что требуется

1. Выполнить EDA: пропуски, дубликаты, баланс классов, распределения ключевых признаков.
2. Разделить данные 80/20 со стратификацией.
3. Собрать `Pipeline`: `SimpleImputer` → `RandomForestClassifier(bootstrap=True, oob_score=True)`.
4. Через `GridSearchCV` подобрать `n_estimators`, `max_features`, `max_depth`, `min_samples_leaf`, `class_weight`. Основная метрика — ROC-AUC.
5. На test вычислить ROC-AUC, accuracy, precision, recall и F1 именно для злокачественного класса.
6. Сопоставить CV ROC-AUC, OOB ROC-AUC и test ROC-AUC; объяснить, почему они не обязаны совпадать.
7. Построить OOB-error curve по числу деревьев.
8. Сравнить impurity importance с permutation importance.
9. Выполнить отбор признаков **внутри Pipeline** через `SelectFromModel`, чтобы избежать утечки, и оценить его через CV.

## Контрольные вопросы

1. Почему OOB доступна только при `bootstrap=True`?
2. Почему OOB нестабильна при малом числе деревьев?
3. Может ли OOB полностью заменить независимую test-выборку?
4. Почему recall злокачественного класса может быть важнее общей accuracy?

In [ ]:
# 5.1 Загрузка, перекодировка target и EDA
cancer = load_breast_cancer(as_frame=True)
X5 = cancer.data.copy()
y5 = (cancer.target == 0).astype(int).rename("malignant")

display(X5.head())
print("1 = malignant; 0 = benign")
print(y5.value_counts())

# TODO: проверки и визуализации баланса/ключевых признаков.

In [ ]:
# 5.2 Pipeline и GridSearchCV
# TODO: stratified split.
# TODO: RandomForestClassifier(bootstrap=True, oob_score=True).
# TODO: сетка n_estimators, max_features, max_depth,
# min_samples_leaf, class_weight; scoring="roc_auc".

In [ ]:
# 5.3 Test, OOB и пороги
# TODO: ROC-AUC, accuracy, precision, recall, F1 для malignant=1.
# TODO: confusion matrix, ROC и PR curves.
# TODO: ROC-AUC по oob_decision_function_ и сравнение CV/OOB/test.
# TODO: исследуйте пороги только на OOF-прогнозах train, не на test.

In [ ]:
# 5.4 OOB curve и важности
# TODO: OOB error для нескольких n_estimators.
# TODO: impurity и permutation importance.

In [ ]:
# 5.5 Отбор признаков без утечки
# TODO: Pipeline: imputer -> SelectFromModel(RandomForestClassifier) -> model.
# TODO: оцените весь Pipeline через cross_val_score на train.
# TODO: после fit получите selected feature names и один раз оцените test.

### Вывод и ответы к задаче 5

**Ваш вывод:** …

1. …
2. …
3. …
4. …

# Чек-лист перед сдачей

- [ ] Все ячейки выполняются сверху вниз без ошибок.
- [ ] Во всех пяти задачах test не использовался при подборе параметров.
- [ ] Для классификации применена стратификация.
- [ ] Есть `Pipeline` и `GridSearchCV` в каждой задаче.
- [ ] Масштабирование либо обоснованно не применено, либо выполнено внутри `Pipeline`.
- [ ] Есть численные метрики и визуальный анализ ошибок.
- [ ] Важности не интерпретируются как причинность.
- [ ] Даны ответы на все 20 контрольных вопросов.